# Marcin Oleszczuk,
laboratorium nr. 2,
uczenie maszynowe 1,
nr. albumu 39385

In [1]:
import pandas
import numpy as np
import time
from scipy import sparse
from sklearn.datasets import fetch_rcv1

In [2]:
def freq(X, prob=True):
    xi, ni = np.unique(X)
    return xi, (ni if not prob else ni / len(X))

In [3]:
def freq2(X, Y, prob=True):
    xi = np.unique(X)
    yj = np.unique(Y)
    x_sparse = False
    y_sparse = False
    if sparse.issparse(X):
        x_sparse = True
        if X.dtype == "bool":
            xi = np.array([True, False])
        else:
            raise AttributeError()
    if sparse.issparse(Y):
        y_sparse = True
        if Y.dtype == "bool":
            yj = np.array([True, False])
        else:
            raise AttributeError()

    nij = np.zeros((xi.size, yj.size))
    if y_sparse and x_sparse:
        joined = sparse.hstack((X.astype('int32'), Y.astype('int32')))
        df = pandas.DataFrame(joined.todense())
        df["D"] = np.zeros([X.shape[0], 1])
        df.columns = ['X', 'Y', 'D']
        pt = pandas.pivot_table(df, values=['D'], index=['X', 'Y'], aggfunc={'D': 'count'})
        flattened = pandas.DataFrame(pt.to_records())
        for index, row in flattened.iterrows():
            nij[row['X'], row['Y']] = row['D']
    else:
        for i, x in enumerate(xi):
            for j, y in enumerate(yj):
                tested_and = np.logical_and(X == x, Y == y)
                nij[i, j] = np.sum(tested_and)

    if prob:
        nij = nij / np.sum(nij)

    return xi, yj, nij

In [4]:
def entropy(p):
    return -np.sum(p * np.log2(p + 1e-16))


def infogain(p):
    return entropy(np.sum(p, axis=1)) + entropy(np.sum(p, axis=0)) - entropy(p)


def kappa(p):
    return infogain(p) / entropy(p)


def gini(p):
    return 1 - np.sum(p ** 2)


def ginigain(p):
    return gini(np.sum(p, axis=1)) + gini(np.sum(p, axis=0)) - gini(p)

In [10]:
def zooAttSelection(df):
    info_type_zoo = df['type']
    columns_relevance = {}
    for i in range(1, 17):
        attributes = df.iloc[:, i]
        xi, yj, pij = freq2(attributes, info_type_zoo)
        k = kappa(pij)
        ig = infogain(pij)
        g = ginigain(pij)
        columns_relevance[df.columns[i]] = (k, g, ig)

    relations = pandas.DataFrame.from_dict(columns_relevance, orient='index',
                                           columns=['kappa', 'ginigain', 'infogain'])
    print(relations)

In [11]:
def wordInfoGain(X, Y):
    columns_relevance = {}
    start = time.time()
    for i in range(0, X.shape[1]):
        attributes = X[:, i]
        xi, yj, pij = freq2(attributes, Y)
        k = kappa(pij)
        ig = infogain(pij)
        g = ginigain(pij)
        columns_relevance[i] = (k, g, ig)

    end = time.time()
    relations = pandas.DataFrame.from_dict(columns_relevance, orient='index', columns=['kappa', 'ginigain', 'infogain'])
    print('Whole operation for ', X.shape[1], 'words took: ', end - start)
    print(relations)
    get50Mostrelevant(relations)


def get50Mostrelevant(relations):
    biggestInfoGain = relations.nlargest(50, 'infogain')
    print('50 words with biggest info gain: \n', biggestInfoGain)

In [12]:
df = pandas.read_csv('./zoo.csv')
zooAttSelection(df)
rcv1 = fetch_rcv1(subset='train')
X = rcv1["data"]
Y = rcv1.target[:, 87]
X = X.astype("bool")
Y = Y.astype("bool")
wordInfoGain(X, Y)

             kappa  ginigain  infogain
hair      0.305999  0.470542  0.790675
feathers  0.300327  0.317616  0.717950
eggs      0.326841  0.475444  0.830138
milk      0.407570  0.482306  0.974320
airborne  0.173199  0.332124  0.469703
aquatic   0.132445  0.395647  0.389487
predator  0.028416  0.379767  0.093447
toothed   0.347187  0.469758  0.865694
backbone  0.282847  0.292912  0.676163
breathes  0.244472  0.324478  0.614494
venomous  0.050093  0.136261  0.133090
fins      0.181013  0.250956  0.466614
legs      0.445247  0.651505  1.363047
tail      0.184472  0.338790  0.500460
domestic  0.017509  0.158808  0.050669
catsize   0.100483  0.409176  0.308490
Whole operation for  47236 words took:  577.4672787189484
          kappa  ginigain  infogain
0      0.000010  0.000003  0.000003
1      0.000052  0.000017  0.000013
2      0.000052  0.000017  0.000013
3      0.000052  0.000017  0.000013
4      0.000052  0.000017  0.000013
...         ...       ...       ...
47231  0.000010  0.000003  

Wnioski:
Na podstawie kolumny sport, najbardziej znaczacy wskaźnik posiada niskie prawdopodobieństwo i niosący ze sobą przyrost informacji (na poziomie 6%).
Do celów obliczeń statystycznych wydaje jest przechowywanie danych w formacie DataFrame, ponieważ umożliwia wiele zoptymalizowanych obliczeń, znacznie przyspieszających proces obróbki danych. Naiwne podejście do problemu, bez skorzystania z pakietu pandas, generowało czas wyliczania funkcji freq2 na poziomie 50 sekund, po skorzystaniu z opisywanego pakietu udało się uzyskać czas pojedyńczego obliczenia na pozomie 0.01 sekundy. Pomimo tego tylko ten treningowy zbiór danych w całości liczył się ok. 12 minut. Dla znacznie większych zbiorów danych sugerowanym podejściem byłoby zwrónoleglenie obliczeń, co pozwoliłoby na znaczne zmniejszenie czasu obliczeń.
Entropia i index Gi
niego niosą ze sobą podobny poziom informacji. Entropia ze względu na istniejący logarytm potrafi dostarczyć dokładniejszych informacji, jednak kosztem czasu obliczeń. Metody te używane są jednak zamiennie i posiadają nie wielkie obszary, w których zastosowanie jednej przynosi lepsze wyniki od drugiej.